# Practical 09: Missing Value Imputation with SimpleImputer & Categorical Encoding

**UNIT 3: Preparing Big Data for Machine Learning**

**Problem Statement:** Write a Python program using Scikit-Learn to preprocess a machine learning dataset: handle missing numerical values using SimpleImputer with the mean strategy, transform categorical text labels into ordinal integers using LabelEncoder, and convert nominal variables to binary dummy indicators using One-Hot Encoding.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

gen = np.random.default_rng(7)
n = 50
age = gen.integers(20, 58, n).astype(float)
salary = np.round(24000 + age * 850 + gen.normal(0, 5500, n), -2)
orders = pd.DataFrame({
    "Country": gen.choice(["France", "Spain", "Germany"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + gen.normal(0, 8000, n) > 54000, "Yes", "No"),
})
orders.loc[gen.choice(n, 5, replace=False), "Age"] = np.nan
orders.loc[gen.choice(n, 5, replace=False), "Salary"] = np.nan
print(orders.head(8))

print(orders.isna().sum())

imputer = SimpleImputer(missing_values=np.nan, strategy="median")
orders[["Age", "Salary"]] = imputer.fit_transform(orders[["Age", "Salary"]])

print("Learned medians (Age, Salary):", imputer.statistics_.round(2))
print("\nMissing values after imputation:")
print(orders.isna().sum())

le_target = LabelEncoder()
orders["Purchased_Encoded"] = le_target.fit_transform(orders["Purchased"])
print("Purchased mapping:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

orders["Country_Label"] = LabelEncoder().fit_transform(orders["Country"])
print("\nNote: Country_Label implies an artificial order (France < Germany < Spain),")
print("so LabelEncoder is unsuitable for nominal features in linear models.")
print(orders[["Country", "Country_Label", "Purchased", "Purchased_Encoded"]].head())

ohe = OneHotEncoder(sparse_output=False, dtype=int)
encoded = ohe.fit_transform(orders[["Country"]])
ohe_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(["Country"]), index=orders.index)

final_df = pd.concat([orders.drop(columns=["Country", "Country_Label"]), ohe_df], axis=1)
print(final_df.head(8))

print(pd.get_dummies(orders["Country"], prefix="Country", dtype=int).head())

,Country,Age,Salary,Purchased
0,Spain,55.0,69700.0,Yes
1,Germany,43.0,46700.0,No
2,Spain,45.0,59300.0,Yes
3,Germany,54.0,69600.0,Yes
4,Germany,41.0,59500.0,No
5,Spain,49.0,57200.0,No
6,France,51.0,64700.0,Yes
7,France,28.0,42400.0,No


Country      0
Age          5
Salary       5
Purchased    0
dtype: int64
Learned medians (Age, Salary): [3.80e+01 5.72e+04]

Missing values after imputation:
Country      0
Age          0
Salary       0
Purchased    0
dtype: int64
Purchased mapping: {'No': np.int64(0), 'Yes': np.int64(1)}

Note: Country_Label implies an artificial order (France < Germany < Spain),
so LabelEncoder is unsuitable for nominal features in linear models.

,Country,Country_Label,Purchased,Purchased_Encoded
0,Spain,2,Yes,1
1,Germany,1,No,0
2,Spain,2,Yes,1
3,Germany,1,Yes,1
4,Germany,1,No,0


,Age,Salary,Purchased,Purchased_Encoded,Country_France,Country_Germany,Country_Spain
0,55.0,69700.0,Yes,1,0,0,1
1,43.0,46700.0,No,0,0,1,0
2,45.0,59300.0,Yes,1,0,0,1
3,54.0,69600.0,Yes,1,0,1,0
4,41.0,59500.0,No,0,0,1,0
5,49.0,57200.0,No,0,0,0,1
6,51.0,64700.0,Yes,1,1,0,0
7,28.0,42400.0,No,0,1,0,0


,Country_France,Country_Germany,Country_Spain
0,0,0,1
1,0,1,0
2,0,0,1
3,0,1,0
4,0,1,0
